# Section 2 — Foundations of AI Pipelines

## Notebook 2 · The training pipeline

*Estimated reading + run time: 50 minutes. Runnable code cells throughout.*

By the end of this notebook you will have:

- Trained four model families on the prepared heart-failure data — logistic regression, random forest, gradient boosting, and SVM — and compared them honestly.
- Selected a single best model and a single deployment threshold, both tuned on the validation set.
- Touched the test set *exactly once* for the final performance estimate.
- Saved the trained model + threshold as `best_heart_failure_model.joblib` for the inference pipeline (notebook 03).

**Inputs:** `prepared_heart_failure.joblib` from notebook 01. If that file is not present (e.g. you skipped notebook 01), this notebook re-builds it from scratch using the same code as notebook 01.

## 1. What this stage owns

The training pipeline takes prepared features and produces a trained, evaluated, threshold-tuned model. Five jobs:

| Job | What it does |
| --- | --- |
| **Model selection** | Train multiple model families — they have different strengths on different clinical problems. |
| **Validation evaluation** | Compare them on metrics that matter clinically (AUC, sensitivity, specificity), not on a single headline number. |
| **Threshold tuning** | Choose the probability cut-off appropriate to the deployment use case. |
| **Single-shot test** | Touch the test set *once*, at the end. Anything else and the test estimate is no longer fair. |
| **Artefact + audit** | Save the model, threshold, and a small summary of which choice won — so the inference pipeline can serve it and a future audit can interrogate it. |

## 2. Load the prepared artefact

Try to load notebook 01's saved artefact; if not present, rebuild it from scratch. The rebuild path mirrors notebook 01 exactly so that running this notebook on its own still works.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

ARTEFACT_PATH = "prepared_heart_failure.joblib"

if os.path.exists(ARTEFACT_PATH):
    artefact = joblib.load(ARTEFACT_PATH)
    print(f"Loaded prepared artefact from {ARTEFACT_PATH}")
else:
    # Rebuild from scratch — mirror of notebook 01.
    print(f"{ARTEFACT_PATH} not found — rebuilding from scratch (mirror of notebook 01).")
    try:
        from datasets import load_dataset
        df = load_dataset("mstz/heart_failure", split="train").to_pandas()
        source = "HuggingFace"
    except Exception:
        df = pd.read_csv(
            "https://archive.ics.uci.edu/ml/machine-learning-databases/00519/"
            "heart_failure_clinical_records_dataset.csv"
        )
        source = "UCI fallback"
    TARGET = "DEATH_EVENT" if "DEATH_EVENT" in df.columns else "is_dead"
    TIME_COL = "time" if "time" in df.columns else "TIME"
    df = df.drop(columns=[TIME_COL])
    y = df[TARGET]
    X = df.drop(columns=[TARGET]).select_dtypes(include="number")
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)
    scaler = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
    X_val_s   = pd.DataFrame(scaler.transform(X_val),   columns=X.columns, index=X_val.index)
    X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=X.columns, index=X_test.index)
    artefact = {
        "X_train": X_train_s, "X_val": X_val_s, "X_test": X_test_s,
        "y_train": y_train.reset_index(drop=True),
        "y_val":   y_val.reset_index(drop=True),
        "y_test":  y_test.reset_index(drop=True),
        "scaler": scaler, "feature_names": list(X.columns),
        "target_name": TARGET, "removed_columns": [TIME_COL], "data_source": source,
    }
    joblib.dump(artefact, ARTEFACT_PATH)
    print(f"Built and saved artefact to {ARTEFACT_PATH}")

X_train, X_val, X_test = artefact["X_train"], artefact["X_val"], artefact["X_test"]
y_train, y_val, y_test = artefact["y_train"], artefact["y_val"], artefact["y_test"]
feature_names = artefact["feature_names"]

print(f"\nTrain: {len(X_train):3d} patients")
print(f"Val:   {len(X_val):3d} patients")
print(f"Test:  {len(X_test):3d} patients")
print(f"Features ({len(feature_names)}): {feature_names}")

## 3. Train four model families

*Why four?* Different model families have different strengths on different clinical problems. The training pipeline trains all of them, compares honestly, and picks the one that wins on the metrics that matter for *this* deployment.

| Model | Strengths | Weaknesses |
| --- | --- | --- |
| **Logistic regression** | Interpretable coefficients; fast; calibrated by default; sensitive to feature scale (which is why notebook 01 standardised). | Limited to linear decision boundaries. Misses interactions between features. |
| **Random forest** | Captures non-linear patterns; robust to feature scaling; handles missing data; gives feature importances. | Less interpretable than logistic regression; can overfit with too many trees. |
| **Gradient boosting** | Often the strongest tabular model on small clinical datasets; built-in regularisation. | Slower to train; many hyperparameters; less calibrated than logistic regression by default. |
| **Support vector machine** | Strong on small high-dimensional datasets; principled margin-based classifier. | Slower on large datasets; needs probability calibration as a separate step. |

*XGBoost* is the live-event default in some places — sklearn's `GradientBoostingClassifier` covers the same teaching ground without an extra dependency, so the booklet uses it instead. If you want XGBoost, install with `pip install xgboost` and swap the line below; the rest of the notebook is unchanged.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# class_weight='balanced' is the imbalance-handling decision (notebook 01, §5).
# It tells the model to penalise mistakes on the minority class proportionally more.
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=5000, class_weight="balanced", random_state=42),
    "Random Forest":       RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(
        n_estimators=200, random_state=42),  # no class_weight; we set thresholds later
    "SVM (RBF)":           SVC(
        probability=True, class_weight="balanced", random_state=42),
}

trained = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    print(f"  ✓ {name} trained")

print(f"\n{len(trained)} models trained on {len(X_train)} patients × {X_train.shape[1]} features.")

## 4. Compare on the validation set — AUC, sensitivity, specificity

Per Chapter 06, *one number is never enough*. The comparison reports AUC plus sensitivity and specificity at the default threshold of 0.50 (we will tune the threshold properly in §6).

★ **Pause-and-predict.** Before running, predict: which model do you expect to win on AUC? On sensitivity?

In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate_at_threshold(y_true, y_proba, threshold=0.50):
    pred = (y_proba >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    sensitivity = tp / (tp + fn) if (tp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0
    return sensitivity, specificity

results = []
for name, model in trained.items():
    y_proba_val = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_proba_val)
    sens, spec = evaluate_at_threshold(y_val, y_proba_val, threshold=0.50)
    results.append({"model": name, "val_auc": auc, "val_sens@0.50": sens, "val_spec@0.50": spec})

results_df = pd.DataFrame(results).sort_values("val_auc", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False, float_format='{:.3f}'.format))

Three things to read from that table. The AUC ranking gives you a rough first cut at which model is strongest. The sensitivity-at-0.50 column is what each model would actually deliver in deployment if you used the default threshold; sometimes the AUC winner is *not* the sensitivity winner, which matters clinically. And the gap between AUC and sensitivity-at-threshold tells you whether threshold tuning will help (a big gap means yes, a small gap means the model is already calibrated near 0.50).

## 5. Visualise — ROC curves all in one frame

ROC curves are the *clinical* way to read this comparison. Each curve traces out the (1 − specificity, sensitivity) trade-off across all thresholds. The model whose curve hugs the top-left corner most tightly wins.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(7, 5.5))
for name, model in trained.items():
    y_proba_val = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_proba_val)
    auc = roc_auc_score(y_val, y_proba_val)
    ax.plot(fpr, tpr, label=f"{name}  (AUC = {auc:.2f})", linewidth=1.8)

ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=0.8, label="Chance line")
ax.set_xlabel("1 − specificity (false-positive rate)")
ax.set_ylabel("Sensitivity (true-positive rate)")
ax.set_title("ROC curves on the validation set — heart-failure mortality")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 6. Threshold tuning — picking the deployment cut-off

Pick the model with the highest validation AUC, sweep the decision threshold across (0.05, 0.95), and pick the threshold that maximises *Youden's J* (sensitivity + specificity − 1) — a balanced choice. **In a real deployment, the threshold is a clinical decision** (Chapter 06): screening tools want lower thresholds, confirmatory tools want higher ones. Youden's J is the demonstration baseline; the clinical team owns the actual choice.

In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = trained[best_name]
y_proba_val = best_model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.05, 0.95, 19)
metrics = []
for t in thresholds:
    sens, spec = evaluate_at_threshold(y_val, y_proba_val, threshold=t)
    youden = sens + spec - 1
    metrics.append({"threshold": t, "sensitivity": sens, "specificity": spec, "youden_j": youden})
metrics_df = pd.DataFrame(metrics)

best_threshold_row = metrics_df.loc[metrics_df["youden_j"].idxmax()]
best_threshold = best_threshold_row["threshold"]

print(f"Best model on validation:        {best_name}")
print(f"Recommended threshold (Youden):  {best_threshold:.2f}")
print(f"  At this threshold:")
print(f"    sensitivity = {best_threshold_row['sensitivity']:.3f}")
print(f"    specificity = {best_threshold_row['specificity']:.3f}")

print("\nFull threshold sweep (every 4th):")
print(metrics_df.iloc[::4].to_string(index=False, float_format='{:.3f}'.format))

**The recommended threshold is the one a balanced-trade-off deployment would default to.** A real deployment would also consider:

- *What is the cost of a false negative?* — for a mortality model, missing a death-bound patient is worse than over-flagging — pull the threshold lower (toward higher sensitivity).
- *What is the cost of a false positive?* — if every flagged patient triggers ICU consideration, the operational cost of false positives is real — pull the threshold higher.
- *What is the prevalence in the deployment population?* — a threshold tuned on training prevalence will produce different (PPV, NPV) at deployment prevalence.

Section 3 Challenge 3 walks the reader through this *as a clinical decision* with a SHAP-explained per-patient view. Here we are setting the demonstration baseline.

## 7. Final test-set evaluation — touch the test set once

Everything above used training and validation only. The test set has not been touched. Now we evaluate the chosen model + threshold on the test set, *exactly once*, and accept whatever number comes back. **If the result is disappointing, the response is not to re-tune.** It is either to accept the result, or — if the result is too poor for deployment — to start over with a fresh test split. The test set must remain a fair test.

In [ ]:
y_proba_test = best_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_proba_test)
test_sens, test_spec = evaluate_at_threshold(y_test, y_proba_test, threshold=best_threshold)

print("=" * 50)
print(f"FINAL TEST-SET EVALUATION ({best_name})")
print("=" * 50)
print(f"  Threshold:    {best_threshold:.2f}")
print(f"  Test AUC:     {test_auc:.3f}")
print(f"  Test sens:    {test_sens:.3f}")
print(f"  Test spec:    {test_spec:.3f}")
print("=" * 50)
print()
print("Compare with validation:")
val_row = results_df[results_df['model'] == best_name].iloc[0]
print(f"  Validation AUC: {val_row['val_auc']:.3f}")
print(f"  Test AUC:       {test_auc:.3f}")
print(f"  Gap:            {val_row['val_auc'] - test_auc:+.3f}")
if abs(val_row['val_auc'] - test_auc) > 0.10:
    print("  ⚠ Large gap between validation and test → look for overfitting or split issues.")
else:
    print("  ✓ Small gap between validation and test → the model is generalising.")

**A small gap between validation and test means you tuned without over-tuning.** A large gap means the validation set has been peeked at too many times — the test estimate is more trustworthy than the validation estimate, and the deployment claim should rest on the test number, not the higher validation one.

## 8. Save the trained model

Save the model + threshold + feature names + decision-trail metadata into a single artefact the inference pipeline (notebook 03) will load. *Pair the model with everything the deployment needs to use it correctly* — version, threshold, feature names, expected scaling. Notebook 01 saved the scaler; notebook 02 saves the model + threshold.

In [ ]:
model_artefact = {
    "model":          best_model,
    "model_name":     best_name,
    "threshold":      float(best_threshold),
    "feature_names":  feature_names,
    "target_name":    artefact["target_name"],
    "validation_metrics": {
        "auc":         float(val_row["val_auc"]),
        "sensitivity": float(best_threshold_row["sensitivity"]),
        "specificity": float(best_threshold_row["specificity"]),
    },
    "test_metrics": {
        "auc":         float(test_auc),
        "sensitivity": float(test_sens),
        "specificity": float(test_spec),
    },
    "training_data_source": artefact["data_source"],
    "removed_columns":      artefact["removed_columns"],
    "version":              "v1.0",
}

out_path = "best_heart_failure_model.joblib"
joblib.dump(model_artefact, out_path)
size_kb = os.path.getsize(out_path) / 1024
print(f"Saved trained-model artefact → {out_path}  ({size_kb:.0f} KB)")
print()
print("This artefact is what the inference pipeline (notebook 03) will load.")

## 9. What is modifiable at this stage

Like notebook 01's feature pipeline, the training pipeline is a stack of decisions that can be revisited.

| Decision | Asked clinically as | What changes if you re-decide |
| --- | --- | --- |
| Which model families to train | Do I need interpretability or pattern-detection power? | Different family wins, slightly different metrics. |
| Hyperparameters within each family | How aggressively should the model regularise? | Trade-off between fit on training set and generalisation. |
| Class-weighting strategy | How important is catching rare cases vs. avoiding false alarms? | Different sensitivity / specificity at the same threshold. |
| Threshold selection method (Youden's J vs. fixed sensitivity vs. cost-aware) | What is the clinical cost of each error type? | Different deployment behaviour at the same model. |
| When to stop and accept the test number | Is this performance fit for deployment? | A pipeline that *re-tunes after seeing the test set* has produced a marketing number, not a deployment number. |

## What this notebook produced

- `best_heart_failure_model.joblib` — the trained model, the chosen threshold, the validation and test metrics, the source of the training data, and the version stamp.
- The single best-of-four comparison + ROC plot above, which forms the *Performance* slide of the demo-day rubric in Chapter 04 / Demo 2 / Section 3 Challenge 3.

**Next:** notebook 03 — the inference pipeline. Loads the trained model, predicts on a single patient, runs SHAP and LIME for explainability, builds an interactive widget that constructs itself from the SHAP top features (v2.0 lesson), generates a clinical narrative with GPT-4o-mini, and applies clinical guardrails.

---

*Section 2 · Notebook 2 · OCAI Booklet · Spring 2026 Edition*